# 📘 Python Foundations for Data Engineering
## Databricks Notebook — Python Mastery for DE

**What you'll learn:**
- Python fundamentals through a Data Engineering lens
- Strings, lists, dicts, functions — all with DE use cases
- Error handling, OOP, generators, decorators
- Python + Spark integration patterns
- 3 mini projects + 5 interview challenges

**Prerequisites:** Notebook 01 (techmart_dw database must exist)

---

---
# 🏗️ Section 1: Setup & Verification

First, let's confirm our database from Notebook 01 is available.

In [0]:
# Set database context
spark.sql("USE techmart_dw")
print("✅ Using techmart_dw database")

In [0]:
# List all available tables
tables = spark.sql("SHOW TABLES").collect()
print(f"Found {len(tables)} tables:")
for t in tables:
    print(f"  - {t.tableName}")

In [0]:
# Verify row counts
expected = {
    'customers': 5200, 'products': 500, 'orders': 20000,
    'order_items': 50000, 'payments': 22000, 'employees': 200,
    'inventory': 1500, 'shipments': 18000, 'website_events': 100000,
    'app_logs': 50000, 'transactions': 30000
}

print("Table Verification:")
print(f"{'Table':<20} {'Expected':<10} {'Actual':<10} {'Status'}")
print("-" * 55)
for table, exp in expected.items():
    try:
        actual = spark.sql(f"SELECT COUNT(*) AS cnt FROM {table}").collect()[0].cnt
        status = "✅" if actual >= exp else "⚠️"
        print(f"{table:<20} {exp:<10} {actual:<10} {status}")
    except Exception as e:
        print(f"{table:<20} {exp:<10} {'ERROR':<10} ❌ {e}")

---
# 📗 Section 2: Python Foundations (Data Engineering Focus)

## 2.1 Variables & Data Types

**WHY for DE:** Every pipeline config, threshold, and parameter is a variable.
Understanding types prevents silent bugs in data transformations.

**Core types:** `int`, `float`, `str`, `bool`, `None`

In [0]:
# Variables in a DE context
batch_size = 10000          # int: rows per batch
threshold = 0.95            # float: quality threshold
table_name = "customers"    # str: target table
is_incremental = True       # bool: load mode
last_watermark = None       # None: no previous run

# Type checking
print(f"batch_size: {type(batch_size).__name__} = {batch_size}")
print(f"threshold: {type(threshold).__name__} = {threshold}")
print(f"table_name: {type(table_name).__name__} = '{table_name}'")
print(f"is_incremental: {type(is_incremental).__name__} = {is_incremental}")
print(f"last_watermark: {type(last_watermark).__name__} = {last_watermark}")

In [0]:
# Type conversion (common in DE when reading configs)
row_count_str = "50000"
row_count = int(row_count_str)
print(f"Converted '{row_count_str}' to int: {row_count}")

# f-strings for dynamic SQL (THE most important Python skill for DE)
schema = "techmart_dw"
table = "orders"
date_filter = "2024-01-01"
query = f"SELECT COUNT(*) FROM {schema}.{table} WHERE order_date >= '{date_filter}'"
print(f"Generated query: {query}")

# Execute it!
result = spark.sql(query).collect()[0][0]
print(f"Result: {result} rows")

In [0]:
# ============================================
# 🎯 YOUR TURN: Dynamic SQL Query Builder
# ============================================
# Build a dynamic SQL query string using f-strings that:
# 1. Takes a table_name variable = "products"
# 2. Takes a column_name variable = "category"
# 3. Takes a min_price variable = 500
# 4. Generates: SELECT {column}, COUNT(*) FROM {table} WHERE price > {min_price} GROUP BY {column}
# 5. Execute it with spark.sql() and print results
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
table_name = "products"
column_name = "category"
min_price = 500

query = f"SELECT {column_name}, COUNT(*) AS cnt FROM {table_name} WHERE price > {min_price} GROUP BY {column_name}"
print(f"Query: {query}")

results = spark.sql(query).collect()
for row in results:
    print(f"  {row[0]}: {row[1]} products")

## 2.2 Strings (Critical for DE)

**WHY:** Data Engineers spend enormous time cleaning strings — column names, file paths,
table names, SQL queries, log messages. Master string methods = fewer bugs.

**Key methods:** `strip`, `lower`, `replace`, `split`, `join`, `startswith`, `endswith`

In [0]:
# String cleaning for column names
raw_columns = ["  Customer ID ", "First Name", "LAST_NAME", "email-address", "Phone Number "]

cleaned = []
for col in raw_columns:
    clean = col.strip().lower().replace(" ", "_").replace("-", "_")
    cleaned.append(clean)

print("Raw → Cleaned:")
for raw, clean in zip(raw_columns, cleaned):
    print(f"  '{raw}' → '{clean}'")

In [0]:
# split and join: CSV-style operations
csv_line = "1001,John,Doe,john@email.com,Premium"
fields = csv_line.split(",")
print(f"Split: {fields}")

# Rebuild with different separator
pipe_delimited = "|".join(fields)
print(f"Joined: {pipe_delimited}")

# Build SQL column list
columns = ["customer_id", "first_name", "last_name", "email"]
select_clause = ", ".join(columns)
print(f"SELECT {select_clause} FROM customers")

In [0]:
# Path building for data engineering
base_path = "/mnt/data/raw"
source = "orders"
date = "2024-06-01"
file_ext = "parquet"

# f-string path construction
file_path = f"{base_path}/{source}/{date}/{source}_{date.replace('-', '')}.{file_ext}"
print(f"File path: {file_path}")

# startswith/endswith for file filtering
files = ["orders_20240601.parquet", "orders_20240601.csv", "customers_20240601.parquet", "README.md"]
parquet_files = [f for f in files if f.endswith(".parquet")]
order_files = [f for f in files if f.startswith("orders")]
print(f"Parquet files: {parquet_files}")
print(f"Order files: {order_files}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Clean Column Names
# ============================================
# Given this list of messy column names from a CSV:
messy_columns = [
    "  Order ID  ", "ORDER_DATE", "customer-name",
    "Total Amount ($)", "Ship To Address", "IS ACTIVE?"
]
# Clean them to be valid SQL column names:
# - strip whitespace
# - lowercase
# - replace spaces, hyphens, special chars with underscore
# - remove parentheses, dollar signs, question marks
#
# Expected: ['order_id', 'order_date', 'customer_name', 'total_amount_', 'ship_to_address', 'is_active']
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
messy_columns = [
    "  Order ID  ", "ORDER_DATE", "customer-name",
    "Total Amount ($)", "Ship To Address", "IS ACTIVE?"
]

import re
cleaned = []
for col in messy_columns:
    c = col.strip().lower()
    c = re.sub(r'[^a-z0-9_]', '_', c)  # replace non-alphanumeric with _
    c = re.sub(r'_+', '_', c)           # collapse multiple underscores
    c = c.strip('_')                     # remove leading/trailing _
    cleaned.append(c)

print("Cleaned columns:")
for raw, clean in zip(messy_columns, cleaned):
    print(f"  '{raw.strip()}' → '{clean}'")

## 2.3 Lists

**WHY:** Lists are everywhere in DE — column lists, batch records, file paths, partition values.
List comprehensions are the Pythonic way to transform data.

In [0]:
# List basics for DE
columns = ["customer_id", "first_name", "last_name", "email", "phone"]

# Indexing and slicing
print(f"First column: {columns[0]}")
print(f"Last column: {columns[-1]}")
print(f"First 3: {columns[:3]}")

# Append and extend
columns.append("created_at")
columns.extend(["updated_at", "_loaded_at"])
print(f"All columns: {columns}")

# List comprehension: transform
upper_cols = [c.upper() for c in columns]
print(f"Uppercase: {upper_cols}")

# Filtered comprehension
audit_cols = [c for c in columns if c.startswith("_")]
data_cols = [c for c in columns if not c.startswith("_")]
print(f"Audit columns: {audit_cols}")
print(f"Data columns: {data_cols}")

In [0]:
# Build column list from Spark schema
df = spark.table("customers")
schema_cols = [field.name for field in df.schema.fields]
print(f"Customers table has {len(schema_cols)} columns:")
for i, col in enumerate(schema_cols, 1):
    print(f"  {i}. {col}")

# Filter to only string columns
string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
print(f"\nString columns: {string_cols}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Dynamic Column Builder
# ============================================
# 1. Read the schema of the 'orders' table using spark.table("orders").schema.fields
# 2. Build a list of column names that are NOT audit columns (not starting with '_')
# 3. Build a SELECT statement string from those columns
# 4. Execute it and show 5 rows
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
df = spark.table("orders")
all_cols = [f.name for f in df.schema.fields]
data_cols = [c for c in all_cols if not c.startswith("_")]
select_sql = f"SELECT {', '.join(data_cols)} FROM orders LIMIT 5"
print(f"Query: {select_sql}")
spark.sql(select_sql).show()

### List Edge Cases & Performance

Understanding list internals helps you write faster code:

| Operation | Time Complexity | Notes |
|-----------|----------------|-------|
| `list[i]` | O(1) | Direct index access |
| `list.append(x)` | O(1) amortized | Occasional resize |
| `list.insert(0, x)` | O(n) | Shifts ALL elements! |
| `list.pop()` | O(1) | Remove from end |
| `list.pop(0)` | O(n) | Shifts ALL elements! |
| `x in list` | O(n) | Linear search |
| `list.sort()` | O(n log n) | Timsort |
| `list + list` | O(n) | Creates new list |

> 💡 **DE Tip:** If you need fast insert/remove from both ends, use `collections.deque` instead.

In [0]:
# List performance demonstration for DE
import time

# ⚠️ Common mistake: using list.insert(0, x) in a loop
# This is O(n²) total — VERY slow for large lists!

def slow_prepend(n):
    """O(n²) — inserting at front shifts everything."""
    result = []
    for i in range(n):
        result.insert(0, i)  # O(n) each time!
    return result

def fast_append_reverse(n):
    """O(n) — append then reverse once."""
    result = []
    for i in range(n):
        result.append(i)  # O(1) each time
    result.reverse()  # O(n) once
    return result

# Compare
for size in [1000, 5000, 10000]:
    start = time.perf_counter()
    slow_prepend(size)
    slow_time = time.perf_counter() - start
    
    start = time.perf_counter()
    fast_append_reverse(size)
    fast_time = time.perf_counter() - start
    
    print(f"n={size:>6}: insert(0,x)={slow_time:.4f}s | append+reverse={fast_time:.6f}s | "
          f"speedup={slow_time/max(fast_time, 0.000001):.0f}x")

print("\n💡 Lesson: Never use insert(0, x) in a loop. Append + reverse instead.")


### Slicing — The Swiss Army Knife

Slicing is one of Python's most powerful features for data manipulation:

In [0]:
# Slicing patterns every DE should know
data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

print("📊 SLICING PATTERNS")
print("=" * 50)
print(f"Original: {data}")
print()

# Basic slicing
print(f"data[2:5]     = {data[2:5]}")       # Elements 2,3,4
print(f"data[:3]      = {data[:3]}")         # First 3
print(f"data[-3:]     = {data[-3:]}")        # Last 3
print(f"data[::2]     = {data[::2]}")        # Every 2nd element
print(f"data[::-1]    = {data[::-1]}")       # Reversed
print(f"data[1::2]    = {data[1::2]}")       # Odd-indexed elements

# DE-specific patterns
print("\n📌 DE-Specific Patterns:")
batch = list(range(100))
chunk_size = 10
chunks = [batch[i:i+chunk_size] for i in range(0, len(batch), chunk_size)]
print(f"Chunking 100 items into batches of 10: {len(chunks)} chunks")
print(f"First chunk: {chunks[0]}")
print(f"Last chunk: {chunks[-1]}")

# Sliding window
window_size = 3
windows = [data[i:i+window_size] for i in range(len(data) - window_size + 1)]
print(f"\nSliding window (size=3): {windows[:5]}...")


In [0]:
# ============================================
# 🎯 YOUR TURN: List Operations
# ============================================
# Given this list of daily revenue values:
revenue = [1200, 1500, 980, 2100, 1800, 1650, 2300, 1900, 2500, 1100, 1750, 2000]

# 1. Get the last 7 days of revenue
# 2. Calculate the 3-day moving average (use slicing + list comprehension)
# 3. Find all days where revenue was above the overall average
# 4. Split into weeks (chunks of 7)
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
revenue = [1200, 1500, 980, 2100, 1800, 1650, 2300, 1900, 2500, 1100, 1750, 2000]

# 1. Last 7 days
last_7 = revenue[-7:]
print(f"1. Last 7 days: {last_7}")

# 2. 3-day moving average
moving_avg = [sum(revenue[i:i+3])/3 for i in range(len(revenue)-2)]
print(f"2. 3-day moving avg: {[round(x, 1) for x in moving_avg]}")

# 3. Days above average
avg = sum(revenue) / len(revenue)
above_avg = [(i+1, r) for i, r in enumerate(revenue) if r > avg]
print(f"3. Average: {avg:.0f}, Days above: {above_avg}")

# 4. Split into weeks
weeks = [revenue[i:i+7] for i in range(0, len(revenue), 7)]
print(f"4. Weeks: {weeks}")


## 2.4 Dictionaries

**WHY:** Dicts are the backbone of pipeline configs, schema mappings, and metadata.
Every JSON config file becomes a dict in Python.

In [0]:
# Pipeline configuration as a dict
pipeline_config = {
    "name": "orders_pipeline",
    "source": {
        "database": "techmart_dw",
        "table": "orders",
        "filter": "status = 'completed'"
    },
    "target": {
        "database": "techmart_dw",
        "table": "gold_daily_sales"
    },
    "settings": {
        "batch_size": 10000,
        "mode": "incremental",
        "watermark_column": "created_at"
    }
}

# Access nested values
print(f"Pipeline: {pipeline_config['name']}")
print(f"Source table: {pipeline_config['source']['table']}")
print(f"Mode: {pipeline_config['settings']['mode']}")

# Safe access with .get() (returns None instead of KeyError)
retry_count = pipeline_config['settings'].get('retry_count', 3)
print(f"Retry count (default): {retry_count}")

In [0]:
# Dict comprehension: build column type mapping
df = spark.table("customers")
type_map = {f.name: str(f.dataType) for f in df.schema.fields}
print("Column type mapping:")
for col, dtype in list(type_map.items())[:8]:
    print(f"  {col}: {dtype}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Pipeline Config Builder
# ============================================
# Build a pipeline_config dict with:
# - "name": "customer_360_pipeline"
# - "sources": a list of dicts, each with "table" and "join_key"
#   (customers/customer_id, orders/customer_id, payments/order_id)
# - "target": {"table": "customer_360", "mode": "overwrite"}
# - "quality_checks": {"null_threshold": 0.05, "duplicate_check": True}
# Print the config nicely
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
pipeline_config = {
    "name": "customer_360_pipeline",
    "sources": [
        {"table": "customers", "join_key": "customer_id"},
        {"table": "orders", "join_key": "customer_id"},
        {"table": "payments", "join_key": "order_id"}
    ],
    "target": {"table": "customer_360", "mode": "overwrite"},
    "quality_checks": {"null_threshold": 0.05, "duplicate_check": True}
}

import json
print(json.dumps(pipeline_config, indent=2))

### Dictionary Patterns for Data Engineering

Dictionaries are the MOST used data structure in DE (configs, mappings, aggregations):

In [0]:
# DE-specific dictionary patterns

# Pattern 1: Frequency counting (most common in DE)
events = ["click", "view", "click", "purchase", "view", "click", "view", "view"]
freq = {}
for event in events:
    freq[event] = freq.get(event, 0) + 1
print(f"1. Frequency count: {freq}")

# Better: use collections.Counter
from collections import Counter
freq2 = Counter(events)
print(f"   Counter version: {dict(freq2)}")
print(f"   Top 2: {freq2.most_common(2)}")

# Pattern 2: Grouping records by key
orders = [
    {"id": 1, "customer": "Alice", "amount": 100},
    {"id": 2, "customer": "Bob", "amount": 200},
    {"id": 3, "customer": "Alice", "amount": 150},
    {"id": 4, "customer": "Bob", "amount": 75},
    {"id": 5, "customer": "Alice", "amount": 300},
]

from collections import defaultdict
by_customer = defaultdict(list)
for order in orders:
    by_customer[order["customer"]].append(order)

print(f"\n2. Grouped by customer:")
for customer, customer_orders in by_customer.items():
    total = sum(o["amount"] for o in customer_orders)
    print(f"   {customer}: {len(customer_orders)} orders, ${total} total")

# Pattern 3: Config/mapping dictionaries
column_mapping = {
    "cust_id": "customer_id",
    "ord_dt": "order_date",
    "amt": "amount",
    "stat": "status",
}

raw_record = {"cust_id": 42, "ord_dt": "2024-03-15", "amt": 99.99, "stat": "completed"}
clean_record = {column_mapping.get(k, k): v for k, v in raw_record.items()}
print(f"\n3. Column mapping:")
print(f"   Raw:   {raw_record}")
print(f"   Clean: {clean_record}")

# Pattern 4: Nested dict for hierarchical data
metrics = defaultdict(lambda: defaultdict(float))
data = [
    ("2024-03", "electronics", 5000),
    ("2024-03", "clothing", 3000),
    ("2024-04", "electronics", 6000),
    ("2024-04", "clothing", 3500),
]
for month, category, revenue in data:
    metrics[month][category] += revenue

print(f"\n4. Nested aggregation:")
for month, categories in sorted(metrics.items()):
    print(f"   {month}: {dict(categories)}")


In [0]:
# ============================================
# 🎯 YOUR TURN: Dictionary Challenges
# ============================================
# Given this list of log entries:
logs = [
    {"timestamp": "2024-03-15 10:00", "level": "ERROR", "service": "auth", "message": "Login failed"},
    {"timestamp": "2024-03-15 10:01", "level": "INFO", "service": "orders", "message": "Order created"},
    {"timestamp": "2024-03-15 10:02", "level": "ERROR", "service": "payments", "message": "Timeout"},
    {"timestamp": "2024-03-15 10:03", "level": "WARN", "service": "auth", "message": "Rate limited"},
    {"timestamp": "2024-03-15 10:04", "level": "ERROR", "service": "auth", "message": "Login failed"},
    {"timestamp": "2024-03-15 10:05", "level": "INFO", "service": "orders", "message": "Order shipped"},
    {"timestamp": "2024-03-15 10:06", "level": "ERROR", "service": "payments", "message": "Card declined"},
]

# 1. Count errors per service
# 2. Group logs by level
# 3. Find the service with the most errors
# 4. Create a summary: {service: {level: count}}
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from collections import Counter, defaultdict

# 1. Errors per service
errors_per_service = Counter(
    log["service"] for log in logs if log["level"] == "ERROR"
)
print(f"1. Errors per service: {dict(errors_per_service)}")

# 2. Group by level
by_level = defaultdict(list)
for log in logs:
    by_level[log["level"]].append(log)
print(f"2. Logs by level: { {k: len(v) for k, v in by_level.items()} }")

# 3. Service with most errors
worst_service = errors_per_service.most_common(1)[0]
print(f"3. Worst service: {worst_service[0]} ({worst_service[1]} errors)")

# 4. Summary matrix
summary = defaultdict(lambda: defaultdict(int))
for log in logs:
    summary[log["service"]][log["level"]] += 1
print(f"4. Summary:")
for service, levels in sorted(summary.items()):
    print(f"   {service}: {dict(levels)}")


## 2.5 Tuples & Sets

**Tuples:** Immutable sequences — perfect for configs that shouldn't change.
**Sets:** Unordered unique elements — perfect for deduplication and membership testing.

In [0]:
# Tuples: immutable configs
VALID_STATUSES = ("completed", "shipped", "processing", "pending", "cancelled", "returned")
PARTITION_COLS = ("order_date", "order_source")

print(f"Valid statuses: {VALID_STATUSES}")
print(f"Is 'completed' valid? {'completed' in VALID_STATUSES}")
print(f"Is 'deleted' valid? {'deleted' in VALID_STATUSES}")

# Tuple unpacking
source_db, source_table = "techmart_dw", "orders"
print(f"Source: {source_db}.{source_table}")

In [0]:
# Sets: find new vs existing records
existing_ids = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
incoming_ids = {8, 9, 10, 11, 12, 13, 14, 15}

new_records = incoming_ids - existing_ids        # difference
updated_records = incoming_ids & existing_ids    # intersection
all_records = incoming_ids | existing_ids        # union
deleted_records = existing_ids - incoming_ids    # what's gone

print(f"Existing: {len(existing_ids)} records")
print(f"Incoming: {len(incoming_ids)} records")
print(f"New (INSERT): {new_records}")
print(f"Updated (UPDATE): {updated_records}")
print(f"Deleted (SOFT DELETE): {deleted_records}")
print(f"Total after merge: {len(all_records)}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Set Operations for CDC
# ============================================
# Simulate a CDC (Change Data Capture) scenario:
# 1. Get existing customer_ids from the customers table (first 100)
# 2. Create a "new batch" set with ids 90-110
# 3. Find: new_customers (to INSERT), existing_customers (to UPDATE),
#    and customers not in new batch (unchanged)
# 4. Print counts for each category
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
# Get existing IDs
rows = spark.sql("SELECT customer_id FROM customers WHERE customer_id <= 100").collect()
existing = set(row.customer_id for row in rows)

# Simulate new batch
new_batch = set(range(90, 111))

new_customers = new_batch - existing
to_update = new_batch & existing
unchanged = existing - new_batch

print(f"Existing customers: {len(existing)}")
print(f"New batch size: {len(new_batch)}")
print(f"To INSERT (new): {len(new_customers)} → {sorted(new_customers)}")
print(f"To UPDATE (existing): {len(to_update)}")
print(f"Unchanged: {len(unchanged)}")

## 2.6 Conditionals & Loops

**WHY:** Control flow drives pipeline logic — batch processing, data classification,
retry mechanisms, and conditional transformations.

In [0]:
# Conditionals: classify customers into tiers
sample_customers = spark.sql("""
    SELECT customer_id, lifetime_value, customer_segment
    FROM customers LIMIT 20
""").collect()

for customer in sample_customers[:10]:
    ltv = float(customer.lifetime_value)
    if ltv >= 40000:
        tier = "Platinum"
    elif ltv >= 25000:
        tier = "Gold"
    elif ltv >= 10000:
        tier = "Silver"
    else:
        tier = "Bronze"
    print(f"  Customer {customer.customer_id}: ${ltv:,.0f} → {tier}")

In [0]:
# Loops with enumerate and zip
tables = ["customers", "orders", "products"]
expected_counts = [5200, 20000, 500]

print("Batch verification:")
for i, (table, expected) in enumerate(zip(tables, expected_counts), 1):
    actual = spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    status = "✅ PASS" if actual >= expected else "❌ FAIL"
    print(f"  {i}. {table}: {actual}/{expected} {status}")

In [0]:
# While loop: batch processing simulation
total_records = 50000
batch_size = 10000
offset = 0
batch_num = 0

print(f"Processing {total_records} records in batches of {batch_size}:")
while offset < total_records:
    batch_num += 1
    end = min(offset + batch_size, total_records)
    print(f"  Batch {batch_num}: rows {offset+1} to {end}")
    offset += batch_size

print(f"Done! Processed {batch_num} batches.")

In [0]:
# ============================================
# 🎯 YOUR TURN: Classify Records
# ============================================
# 1. Query the first 15 orders (order_id, total_amount, status)
# 2. Loop through them and classify:
#    - total_amount > 4000 AND status='completed' → "High Priority"
#    - total_amount > 2000 → "Medium Priority"
#    - status = 'cancelled' → "Skip"
#    - else → "Low Priority"
# 3. Count how many in each category
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
orders = spark.sql("SELECT order_id, total_amount, status FROM orders LIMIT 15").collect()

counts = {"High Priority": 0, "Medium Priority": 0, "Low Priority": 0, "Skip": 0}

for order in orders:
    amt = float(order.total_amount)
    if amt > 4000 and order.status == 'completed':
        priority = "High Priority"
    elif amt > 2000:
        priority = "Medium Priority"
    elif order.status == 'cancelled':
        priority = "Skip"
    else:
        priority = "Low Priority"
    counts[priority] += 1
    print(f"  Order {order.order_id}: ${amt:,.0f} ({order.status}) → {priority}")

print(f"\nSummary: {counts}")

## 2.7 Functions

**WHY:** Functions are the building blocks of reusable pipelines.
Well-designed functions make pipelines testable, composable, and maintainable.

In [0]:
# Function with type hints and docstring
def get_table_stats(table_name: str, database: str = "techmart_dw") -> dict:
    """Get basic statistics for a table.
    
    Args:
        table_name: Name of the table to profile
        database: Database name (default: techmart_dw)
    
    Returns:
        Dict with row_count, column_count, and column names
    """
    df = spark.table(f"{database}.{table_name}")
    return {
        "table": table_name,
        "row_count": df.count(),
        "column_count": len(df.columns),
        "columns": df.columns
    }

# Use it
stats = get_table_stats("orders")
print(f"Table: {stats['table']}")
print(f"Rows: {stats['row_count']:,}")
print(f"Columns: {stats['column_count']}")

In [0]:
# Function with *args and **kwargs
def build_query(table: str, *columns, **filters) -> str:
    """Build a SELECT query dynamically.
    
    Args:
        table: Table name
        *columns: Column names to select (default: *)
        **filters: WHERE conditions as key=value pairs
    """
    col_str = ", ".join(columns) if columns else "*"
    query = f"SELECT {col_str} FROM {table}"
    
    if filters:
        conditions = [f"{k} = '{v}'" if isinstance(v, str) else f"{k} = {v}" 
                     for k, v in filters.items()]
        query += " WHERE " + " AND ".join(conditions)
    
    return query

# Examples
print(build_query("orders"))
print(build_query("orders", "order_id", "total_amount", status="completed"))
print(build_query("customers", "customer_id", "email", customer_segment="Premium", is_active=True))

In [0]:
# ============================================
# 🎯 YOUR TURN: Data Validation Function
# ============================================
# Write a function `validate_table(table_name)` that:
# 1. Checks if the table exists (try/except)
# 2. Gets row count
# 3. Checks for NULL primary key (assume first column is PK)
# 4. Returns a dict: {"table", "exists", "row_count", "null_pk_count", "is_valid"}
# 5. is_valid = True if exists AND row_count > 0 AND null_pk_count == 0
#
# Test with "orders" and "nonexistent_table"
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
def validate_table(table_name: str) -> dict:
    """Validate a table's basic integrity."""
    result = {"table": table_name, "exists": False, "row_count": 0, "null_pk_count": 0, "is_valid": False}
    
    try:
        df = spark.table(table_name)
        result["exists"] = True
        result["row_count"] = df.count()
        
        pk_col = df.columns[0]
        null_count = df.filter(f"{pk_col} IS NULL").count()
        result["null_pk_count"] = null_count
        result["is_valid"] = result["row_count"] > 0 and null_count == 0
    except Exception as e:
        result["error"] = str(e)
    
    return result

# Test
for t in ["orders", "nonexistent_table"]:
    r = validate_table(t)
    print(f"{t}: {r}")

### Function Patterns for Data Engineering

These patterns appear in EVERY production pipeline:

In [0]:
# Pattern 1: Validation functions
def validate_record(record, schema):
    """
    Validate a record against a schema definition.
    Returns (is_valid, errors) tuple.
    
    This pattern is used in every Bronze → Silver transformation.
    """
    errors = []
    
    for field, rules in schema.items():
        value = record.get(field)
        
        # Required check
        if rules.get("required") and value is None:
            errors.append(f"{field}: required but missing")
            continue
        
        if value is None:
            continue
        
        # Type check
        expected_type = rules.get("type")
        if expected_type and not isinstance(value, expected_type):
            errors.append(f"{field}: expected {expected_type.__name__}, got {type(value).__name__}")
        
        # Range check
        if "min" in rules and value < rules["min"]:
            errors.append(f"{field}: {value} below minimum {rules['min']}")
        if "max" in rules and value > rules["max"]:
            errors.append(f"{field}: {value} above maximum {rules['max']}")
    
    return len(errors) == 0, errors


# Define schema
order_schema = {
    "order_id": {"required": True, "type": int, "min": 1},
    "amount": {"required": True, "type": (int, float), "min": 0, "max": 1000000},
    "customer_id": {"required": True, "type": int},
    "status": {"required": True, "type": str},
}

# Test records
test_records = [
    {"order_id": 1, "amount": 99.99, "customer_id": 42, "status": "completed"},
    {"order_id": None, "amount": 50, "customer_id": 1, "status": "pending"},
    {"order_id": 3, "amount": -10, "customer_id": 5, "status": "failed"},
    {"order_id": 4, "amount": 200, "customer_id": None, "status": "shipped"},
]

print("📋 RECORD VALIDATION")
print("=" * 60)
for record in test_records:
    is_valid, errors = validate_record(record, order_schema)
    status = "✅ VALID" if is_valid else "❌ INVALID"
    print(f"  {status}: {record}")
    if errors:
        for e in errors:
            print(f"         → {e}")


In [0]:
# Pattern 2: Retry decorator (essential for API calls, DB connections)
import time
import random

def retry(max_attempts=3, delay=1, backoff=2, exceptions=(Exception,)):
    """
    Retry decorator with exponential backoff.
    
    Used in production for:
    - API calls (network timeouts)
    - Database connections (transient failures)
    - File operations (lock contention)
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            attempts = 0
            current_delay = delay
            
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    attempts += 1
                    if attempts == max_attempts:
                        raise
                    print(f"    ⚠️ Attempt {attempts} failed: {e}. Retrying in {current_delay}s...")
                    # time.sleep(current_delay)  # Commented for demo speed
                    current_delay *= backoff
            
        return wrapper
    return decorator


# Demo: Simulate a flaky API
call_count = {"n": 0}

@retry(max_attempts=3, delay=1, backoff=2)
def fetch_from_api(endpoint):
    """Simulates an API that fails intermittently."""
    call_count["n"] += 1
    if call_count["n"] <= 2:
        raise ConnectionError(f"Timeout connecting to {endpoint}")
    return {"status": "success", "data": [1, 2, 3]}

print("🔄 RETRY DECORATOR DEMO")
print("=" * 50)
result = fetch_from_api("/api/v1/orders")
print(f"    ✅ Final result: {result}")
print(f"    Total attempts: {call_count['n']}")


In [0]:
# Pattern 3: Pipeline function composition
def compose_pipeline(*functions):
    """
    Compose multiple transformation functions into a pipeline.
    Each function takes data and returns transformed data.
    
    This is the functional programming approach to ETL.
    """
    def pipeline(data):
        result = data
        for func in functions:
            result = func(result)
        return result
    return pipeline


# Define transformation steps
def remove_nulls(records):
    """Remove records with None values in required fields."""
    return [r for r in records if r.get("amount") is not None]

def normalize_status(records):
    """Standardize status values."""
    status_map = {"complete": "completed", "ship": "shipped", "pend": "pending"}
    for r in records:
        r["status"] = status_map.get(r["status"], r["status"]).lower()
    return records

def add_audit_fields(records):
    """Add metadata for tracking."""
    from datetime import datetime
    for r in records:
        r["_processed_at"] = datetime.now().isoformat()
        r["_source"] = "raw_orders"
    return records

def calculate_tax(records):
    """Add tax calculation."""
    for r in records:
        if r.get("amount"):
            r["tax"] = round(r["amount"] * 0.08, 2)
            r["total"] = round(r["amount"] + r["tax"], 2)
    return records


# Compose the pipeline
etl_pipeline = compose_pipeline(
    remove_nulls,
    normalize_status,
    calculate_tax,
    add_audit_fields,
)

# Run it
raw_data = [
    {"id": 1, "amount": 100, "status": "complete"},
    {"id": 2, "amount": None, "status": "pend"},
    {"id": 3, "amount": 250, "status": "ship"},
    {"id": 4, "amount": 75, "status": "complete"},
]

print("🔧 PIPELINE COMPOSITION")
print("=" * 50)
print(f"Input: {len(raw_data)} records")
result = etl_pipeline(raw_data)
print(f"Output: {len(result)} records (1 removed for null amount)")
for r in result:
    print(f"  {r}")


## 2.8 Comprehensions

**WHY:** Comprehensions are concise, readable, and Pythonic. They replace verbose loops
for transforming data. Generator expressions save memory on large datasets.

In [0]:
# List comprehension: transform records
raw_records = [
    {"name": "  JOHN DOE  ", "email": "JOHN@GMAIL.COM", "amount": "1500.50"},
    {"name": "jane smith", "email": "jane@yahoo.com  ", "amount": "2300.00"},
    {"name": "Bob Wilson  ", "email": "  BOB@OUTLOOK.COM", "amount": "890.25"},
]

# Clean all records in one expression
cleaned = [
    {
        "name": r["name"].strip().title(),
        "email": r["email"].strip().lower(),
        "amount": float(r["amount"])
    }
    for r in raw_records
]

for r in cleaned:
    print(f"  {r['name']} | {r['email']} | ${r['amount']:,.2f}")

In [0]:
# Dict comprehension: column type mapping
df = spark.table("orders")
type_map = {f.name: str(f.dataType).replace("Type()", "") for f in df.schema.fields}
print("Orders column types:")
for col, dtype in type_map.items():
    print(f"  {col}: {dtype}")

In [0]:
# Generator expression: memory-efficient for large data
# Instead of creating a full list in memory:
# big_list = [x**2 for x in range(1_000_000)]  # uses lots of memory

# Use a generator (lazy evaluation):
big_gen = (x**2 for x in range(1_000_000))
print(f"Generator object: {big_gen}")
print(f"First 5 values: {[next(big_gen) for _ in range(5)]}")

# Sum without materializing the full list
total = sum(x for x in range(1000))
print(f"Sum of 0-999: {total}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Transform Records
# ============================================
# Given this raw data from an API:
raw_events = [
    {"event": "PAGE_VIEW", "user": "user_101", "ts": "2024-06-01T10:30:00", "value": ""},
    {"event": "click", "user": "USER_102", "ts": "2024-06-01T10:31:00", "value": "45.5"},
    {"event": "PURCHASE", "user": "user_103", "ts": "2024-06-01T10:32:00", "value": "199.99"},
    {"event": "page_view", "user": "USER_104", "ts": "2024-06-01T10:33:00", "value": ""},
]
# Use comprehensions to:
# 1. Normalize event to lowercase
# 2. Normalize user to lowercase
# 3. Convert value to float (0.0 if empty)
# 4. Filter to only events where value > 0
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
raw_events = [
    {"event": "PAGE_VIEW", "user": "user_101", "ts": "2024-06-01T10:30:00", "value": ""},
    {"event": "click", "user": "USER_102", "ts": "2024-06-01T10:31:00", "value": "45.5"},
    {"event": "PURCHASE", "user": "user_103", "ts": "2024-06-01T10:32:00", "value": "199.99"},
    {"event": "page_view", "user": "USER_104", "ts": "2024-06-01T10:33:00", "value": ""},
]

# Transform
cleaned = [
    {"event": e["event"].lower(), "user": e["user"].lower(), 
     "ts": e["ts"], "value": float(e["value"]) if e["value"] else 0.0}
    for e in raw_events
]

# Filter
with_value = [e for e in cleaned if e["value"] > 0]
print(f"All cleaned: {len(cleaned)} events")
print(f"With value > 0: {len(with_value)} events")
for e in with_value:
    print(f"  {e['event']} by {e['user']}: ${e['value']}")

---
# 📗 Section 3: Intermediate Python for DE

## 3.1 Lambda Functions

**WHY:** Lambdas are anonymous one-liner functions. Perfect for quick transformations
with `sorted()`, `map()`, and `filter()`.

In [0]:
# Lambda with sorted: sort records by multiple keys
records = [
    {"name": "Alice", "dept": "Engineering", "salary": 120000},
    {"name": "Bob", "dept": "Sales", "salary": 95000},
    {"name": "Charlie", "dept": "Engineering", "salary": 140000},
    {"name": "Diana", "dept": "Sales", "salary": 110000},
    {"name": "Eve", "dept": "Engineering", "salary": 130000},
]

# Sort by department, then salary descending
sorted_records = sorted(records, key=lambda r: (r["dept"], -r["salary"]))
print("Sorted by dept ASC, salary DESC:")
for r in sorted_records:
    print(f"  {r['dept']:<15} {r['name']:<10} ${r['salary']:,}")

In [0]:
# map and filter with lambda
amounts = [150.0, 2500.0, 89.99, 4200.0, 750.0, 3100.0, 50.0]

# Filter high-value transactions
high_value = list(filter(lambda x: x > 1000, amounts))
print(f"High value (>$1000): {high_value}")

# Apply tax
with_tax = list(map(lambda x: round(x * 1.08, 2), amounts))
print(f"With 8% tax: {with_tax}")

## 3.2 Error Handling

**WHY:** Pipelines WILL fail. Proper error handling means graceful degradation,
useful error messages, and automatic retries.

In [0]:
# try/except/finally pattern
def safe_query(query: str) -> list:
    """Execute a query with error handling."""
    try:
        result = spark.sql(query).collect()
        print(f"✅ Query returned {len(result)} rows")
        return result
    except Exception as e:
        print(f"❌ Query failed: {type(e).__name__}: {e}")
        return []
    finally:
        print("   (query execution complete)")

# Test with valid and invalid queries
safe_query("SELECT COUNT(*) FROM orders")
safe_query("SELECT * FROM nonexistent_table")

In [0]:
# Retry pattern with exponential backoff
import time

def retry_operation(func, max_retries=3, base_delay=1):
    """Retry a function with exponential backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            result = func()
            print(f"  ✅ Success on attempt {attempt}")
            return result
        except Exception as e:
            if attempt == max_retries:
                print(f"  ❌ Failed after {max_retries} attempts: {e}")
                raise
            delay = base_delay * (2 ** (attempt - 1))
            print(f"  ⚠️ Attempt {attempt} failed: {e}. Retrying in {delay}s...")
            time.sleep(delay)

# Demo (simulated failure)
call_count = [0]
def flaky_operation():
    call_count[0] += 1
    if call_count[0] < 3:
        raise ConnectionError("Database connection timeout")
    return "data loaded successfully"

print("Retry demo:")
result = retry_operation(flaky_operation)
print(f"Result: {result}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Safe Parse Function
# ============================================
# Write a function safe_parse(records) that:
# 1. Takes a list of dicts with potentially messy data
# 2. For each record, tries to:
#    - Convert "amount" to float
#    - Convert "quantity" to int
#    - Validate "status" is in ("active", "inactive")
# 3. Returns (valid_records, error_records) tuple
# 4. error_records should include the original record + error message
#
# Test data:
test_records = [
    {"id": 1, "amount": "150.50", "quantity": "3", "status": "active"},
    {"id": 2, "amount": "bad_data", "quantity": "2", "status": "active"},
    {"id": 3, "amount": "200.00", "quantity": "abc", "status": "active"},
    {"id": 4, "amount": "300.00", "quantity": "1", "status": "unknown"},
    {"id": 5, "amount": "99.99", "quantity": "5", "status": "inactive"},
]
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
def safe_parse(records):
    valid, errors = [], []
    valid_statuses = ("active", "inactive")
    
    for rec in records:
        try:
            parsed = {
                "id": rec["id"],
                "amount": float(rec["amount"]),
                "quantity": int(rec["quantity"]),
                "status": rec["status"]
            }
            if parsed["status"] not in valid_statuses:
                raise ValueError(f"Invalid status: {parsed['status']}")
            valid.append(parsed)
        except (ValueError, KeyError) as e:
            errors.append({"record": rec, "error": str(e)})
    
    return valid, errors

test_records = [
    {"id": 1, "amount": "150.50", "quantity": "3", "status": "active"},
    {"id": 2, "amount": "bad_data", "quantity": "2", "status": "active"},
    {"id": 3, "amount": "200.00", "quantity": "abc", "status": "active"},
    {"id": 4, "amount": "300.00", "quantity": "1", "status": "unknown"},
    {"id": 5, "amount": "99.99", "quantity": "5", "status": "inactive"},
]

valid, errors = safe_parse(test_records)
print(f"Valid: {len(valid)} records")
for r in valid:
    print(f"  ✅ {r}")
print(f"\nErrors: {len(errors)} records")
for e in errors:
    print(f"  ❌ ID {e['record']['id']}: {e['error']}")

### Error Handling Patterns for Production Pipelines

In production, errors WILL happen. The question is: how gracefully do you handle them?

In [0]:
# Production error handling patterns

# Pattern 1: Categorize errors (transient vs permanent)
class TransientError(Exception):
    """Temporary error — safe to retry (network, timeout)."""
    pass

class PermanentError(Exception):
    """Permanent error — do NOT retry (bad data, schema mismatch)."""
    pass

class DataQualityError(PermanentError):
    """Data doesn't meet quality standards."""
    pass


def process_record(record):
    """Process a single record with proper error categorization."""
    # Validate
    if record.get("amount") is None:
        raise DataQualityError(f"NULL amount for record {record.get('id')}")
    if record["amount"] < 0:
        raise DataQualityError(f"Negative amount: {record['amount']}")
    
    # Simulate occasional transient failure
    import random
    if random.random() < 0.1:
        raise TransientError("Database connection timeout")
    
    return {"id": record["id"], "processed_amount": record["amount"] * 1.08}


# Pattern 2: Process batch with error isolation
def process_batch_safely(records):
    """
    Process a batch of records, isolating failures.
    Good records proceed, bad records go to dead letter queue.
    """
    results = []
    dead_letter = []
    retryable = []
    
    for record in records:
        try:
            result = process_record(record)
            results.append(result)
        except PermanentError as e:
            dead_letter.append({"record": record, "error": str(e), "type": "permanent"})
        except TransientError as e:
            retryable.append({"record": record, "error": str(e), "type": "transient"})
    
    return {
        "success": results,
        "dead_letter": dead_letter,
        "retryable": retryable,
    }


# Demo
import random
random.seed(42)

batch = [
    {"id": 1, "amount": 100},
    {"id": 2, "amount": None},     # Will fail: DataQualityError
    {"id": 3, "amount": 200},
    {"id": 4, "amount": -50},      # Will fail: DataQualityError
    {"id": 5, "amount": 300},
    {"id": 6, "amount": 150},
]

print("🛡️ PRODUCTION ERROR HANDLING")
print("=" * 60)
outcome = process_batch_safely(batch)
print(f"  ✅ Successful: {len(outcome['success'])} records")
print(f"  ❌ Dead letter (permanent): {len(outcome['dead_letter'])} records")
print(f"  🔄 Retryable (transient): {len(outcome['retryable'])} records")

print("\n  Dead letter queue:")
for item in outcome["dead_letter"]:
    print(f"    Record {item['record']['id']}: {item['error']}")

print("\n💡 Key Pattern: Never let one bad record kill the entire batch!")
print("   Isolate failures, process what you can, retry transients, DLQ permanents.")


## 3.3 Object-Oriented Programming

**WHY:** Classes encapsulate pipeline logic into reusable, testable components.
A `DataQualityChecker` class is more maintainable than scattered functions.

In [0]:
# Pipeline component as a class
class DataQualityChecker:
    """Reusable data quality checker for any table."""
    
    def __init__(self, table_name: str):
        self.table_name = table_name
        self.df = spark.table(table_name)
        self.results = []
    
    def check_nulls(self, column: str, threshold: float = 0.05) -> dict:
        """Check if null rate exceeds threshold."""
        total = self.df.count()
        nulls = self.df.filter(f"{column} IS NULL").count()
        null_rate = nulls / total if total > 0 else 0
        
        result = {
            "check": "null_rate",
            "column": column,
            "null_count": nulls,
            "null_rate": round(null_rate, 4),
            "threshold": threshold,
            "passed": null_rate <= threshold
        }
        self.results.append(result)
        return result
    
    def check_duplicates(self, columns: list) -> dict:
        """Check for duplicate records."""
        total = self.df.count()
        distinct = self.df.select(columns).distinct().count()
        dupes = total - distinct
        
        result = {
            "check": "duplicates",
            "columns": columns,
            "total_rows": total,
            "distinct_rows": distinct,
            "duplicate_count": dupes,
            "passed": dupes == 0
        }
        self.results.append(result)
        return result
    
    def summary(self) -> None:
        """Print summary of all checks."""
        passed = sum(1 for r in self.results if r["passed"])
        total = len(self.results)
        print(f"\n{'='*50}")
        print(f"Quality Report: {self.table_name}")
        print(f"{'='*50}")
        print(f"Checks: {passed}/{total} passed")
        for r in self.results:
            status = "✅" if r["passed"] else "❌"
            print(f"  {status} {r['check']}: {r.get('column', r.get('columns', ''))}")
        print(f"{'='*50}")

# Use it!
checker = DataQualityChecker("customers")
checker.check_nulls("email", threshold=0.10)
checker.check_nulls("phone", threshold=0.05)
checker.check_duplicates(["email"])
checker.summary()

In [0]:
# ============================================
# 🎯 YOUR TURN: Build a TableProfiler Class
# ============================================
# Create a class `TableProfiler` with:
# 1. __init__(self, table_name) - loads the spark table
# 2. row_count() - returns row count
# 3. column_stats(column) - returns min, max, avg, nulls for a numeric column
# 4. value_distribution(column, top_n=5) - returns top N values with counts
# 5. profile() - runs all checks and prints a nice report
#
# Test with the "orders" table
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
class TableProfiler:
    def __init__(self, table_name: str):
        self.table_name = table_name
        self.df = spark.table(table_name)
    
    def row_count(self) -> int:
        return self.df.count()
    
    def column_stats(self, column: str) -> dict:
        stats = spark.sql(f"""
            SELECT MIN({column}) AS min_val, MAX({column}) AS max_val,
                   ROUND(AVG({column}), 2) AS avg_val,
                   SUM(CASE WHEN {column} IS NULL THEN 1 ELSE 0 END) AS null_count
            FROM {self.table_name}
        """).collect()[0]
        return {"min": stats.min_val, "max": stats.max_val, "avg": stats.avg_val, "nulls": stats.null_count}
    
    def value_distribution(self, column: str, top_n: int = 5) -> list:
        rows = spark.sql(f"""
            SELECT {column}, COUNT(*) AS cnt FROM {self.table_name}
            GROUP BY {column} ORDER BY cnt DESC LIMIT {top_n}
        """).collect()
        return [(row[0], row[1]) for row in rows]
    
    def profile(self):
        print(f"\n📊 Profile: {self.table_name}")
        print(f"   Rows: {self.row_count():,}")
        print(f"   Columns: {len(self.df.columns)}")
        print(f"\n   total_amount stats: {self.column_stats('total_amount')}")
        print(f"\n   Top statuses:")
        for val, cnt in self.value_distribution("status"):
            print(f"     {val}: {cnt:,}")

profiler = TableProfiler("orders")
profiler.profile()

### OOP Patterns in Data Engineering

Classes are used extensively in production pipelines for:
- Configuration management
- Pipeline orchestration
- Data validation
- Connection pooling

In [0]:
# Production-style pipeline class
from datetime import datetime
from typing import List, Dict, Any, Optional

class DataPipeline:
    """
    Base class for data pipelines.
    Provides: logging, metrics, error handling, checkpointing.
    
    Every production pipeline at companies like Netflix, Uber, Airbnb
    follows a similar pattern.
    """
    
    def __init__(self, name: str, source: str, target: str):
        self.name = name
        self.source = source
        self.target = target
        self.metrics = {"records_read": 0, "records_written": 0, 
                       "records_failed": 0, "start_time": None, "end_time": None}
        self.errors: List[Dict] = []
    
    def extract(self) -> List[Dict]:
        """Extract data from source. Override in subclass."""
        raise NotImplementedError("Subclass must implement extract()")
    
    def transform(self, data: List[Dict]) -> List[Dict]:
        """Transform data. Override in subclass."""
        raise NotImplementedError("Subclass must implement transform()")
    
    def load(self, data: List[Dict]) -> int:
        """Load data to target. Override in subclass."""
        raise NotImplementedError("Subclass must implement load()")
    
    def run(self) -> Dict[str, Any]:
        """Execute the full ETL pipeline with error handling."""
        self.metrics["start_time"] = datetime.now()
        print(f"🚀 Starting pipeline: {self.name}")
        
        try:
            # Extract
            raw_data = self.extract()
            self.metrics["records_read"] = len(raw_data)
            print(f"   📥 Extracted: {len(raw_data)} records from {self.source}")
            
            # Transform
            clean_data = self.transform(raw_data)
            print(f"   🔄 Transformed: {len(clean_data)} records")
            
            # Load
            written = self.load(clean_data)
            self.metrics["records_written"] = written
            print(f"   📤 Loaded: {written} records to {self.target}")
            
        except Exception as e:
            self.errors.append({"error": str(e), "timestamp": datetime.now().isoformat()})
            print(f"   ❌ Pipeline failed: {e}")
        
        self.metrics["end_time"] = datetime.now()
        self.metrics["duration_seconds"] = (
            self.metrics["end_time"] - self.metrics["start_time"]
        ).total_seconds()
        self.metrics["records_failed"] = self.metrics["records_read"] - self.metrics["records_written"]
        
        return self.metrics


class OrdersPipeline(DataPipeline):
    """Concrete pipeline for processing orders."""
    
    def __init__(self):
        super().__init__(
            name="daily_orders_pipeline",
            source="raw.orders",
            target="silver.clean_orders"
        )
    
    def extract(self) -> List[Dict]:
        # Simulate reading from source
        return [
            {"id": 1, "amount": 100, "status": "COMPLETED", "customer": "  Alice  "},
            {"id": 2, "amount": 200, "status": "shipped", "customer": "Bob"},
            {"id": 3, "amount": None, "status": "pending", "customer": "Charlie"},
            {"id": 4, "amount": 50, "status": "Completed", "customer": "Diana"},
        ]
    
    def transform(self, data: List[Dict]) -> List[Dict]:
        clean = []
        for record in data:
            if record["amount"] is None:
                self.metrics["records_failed"] = self.metrics.get("records_failed", 0) + 1
                continue
            clean.append({
                "id": record["id"],
                "amount": record["amount"],
                "status": record["status"].lower().strip(),
                "customer": record["customer"].strip().title(),
            })
        return clean
    
    def load(self, data: List[Dict]) -> int:
        # Simulate writing to target
        return len(data)


# Run the pipeline
pipeline = OrdersPipeline()
metrics = pipeline.run()

print(f"\n📊 Pipeline Metrics:")
for key, value in metrics.items():
    if key not in ("start_time", "end_time"):
        print(f"   {key}: {value}")


## 3.4 Generators & Iterators

**WHY:** Generators produce values lazily — one at a time. Essential for processing
large datasets without loading everything into memory.

In [0]:
# Date range generator
from datetime import date, timedelta

def date_range(start_date: str, end_date: str, step_days: int = 1):
    """Generate dates between start and end."""
    current = date.fromisoformat(start_date)
    end = date.fromisoformat(end_date)
    while current <= end:
        yield current
        current += timedelta(days=step_days)

# Use it
print("Daily partitions for June 2024:")
for d in date_range("2024-06-01", "2024-06-07"):
    print(f"  {d} → partition: p_{d.strftime('%Y%m%d')}")

In [0]:
# Batch generator for processing large datasets
def batch_generator(total: int, batch_size: int):
    """Generate (offset, limit) tuples for batch processing."""
    for offset in range(0, total, batch_size):
        yield (offset, min(batch_size, total - offset))

# Simulate batch processing
print("Batch plan for 55,000 records (batch_size=10,000):")
for batch_num, (offset, size) in enumerate(batch_generator(55000, 10000), 1):
    print(f"  Batch {batch_num}: offset={offset}, size={size}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Partition Generator
# ============================================
# Create a generator `monthly_partitions(start_year, start_month, num_months)`
# that yields partition strings like "2024-01", "2024-02", etc.
# Handle year rollover (e.g., 2024-12 → 2025-01)
#
# Test: generate 14 months starting from 2024-03
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
def monthly_partitions(start_year: int, start_month: int, num_months: int):
    year, month = start_year, start_month
    for _ in range(num_months):
        yield f"{year}-{month:02d}"
        month += 1
        if month > 12:
            month = 1
            year += 1

print("Monthly partitions (14 months from 2024-03):")
for partition in monthly_partitions(2024, 3, 14):
    print(f"  {partition}")

## 3.5 Decorators

**WHY:** Decorators add behavior to functions without modifying them.
Common in DE: logging, timing, retrying, input validation.

In [0]:
# Timer decorator
import time
from functools import wraps

def timer(func):
    """Measure execution time of a function."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"  ⏱️ {func.__name__} took {elapsed:.2f}s")
        return result
    return wrapper

def log_execution(func):
    """Log function entry and exit."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"  ▶️ Starting {func.__name__}")
        result = func(*args, **kwargs)
        print(f"  ✅ Completed {func.__name__}")
        return result
    return wrapper

@timer
@log_execution
def count_table(table_name: str) -> int:
    return spark.sql(f"SELECT COUNT(*) FROM {table_name}").collect()[0][0]

result = count_table("orders")
print(f"  Count: {result:,}")

In [0]:
# Retry decorator with arguments
def retry(max_attempts=3, delay=1):
    """Decorator that retries on failure."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"  ⚠️ {func.__name__} attempt {attempt} failed: {e}")
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.1)
def fetch_data(table: str):
    return spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]

print(f"Orders count: {fetch_data('orders'):,}")

In [0]:
# More decorator patterns for DE

# Pattern: Timing decorator (measure pipeline step duration)
import time
import functools

def timer(func):
    """Measure execution time of a function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  ⏱️ {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

# Pattern: Logging decorator
def log_step(step_name):
    """Log pipeline step execution."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(f"  ▶️ Starting: {step_name}")
            try:
                result = func(*args, **kwargs)
                print(f"  ✅ Completed: {step_name}")
                return result
            except Exception as e:
                print(f"  ❌ Failed: {step_name} — {e}")
                raise
        return wrapper
    return decorator

# Pattern: Cache decorator (avoid recomputation)
def cache_result(func):
    """Cache function results (memoization)."""
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper


# Demo: Combine decorators in a pipeline
@timer
@log_step("Extract orders")
def extract_orders():
    time.sleep(0.01)  # Simulate I/O
    return [{"id": i, "amount": i * 10} for i in range(1000)]

@timer
@log_step("Transform orders")
def transform_orders(data):
    return [{"id": r["id"], "amount_with_tax": r["amount"] * 1.08} for r in data]

@timer
@log_step("Load orders")
def load_orders(data):
    return len(data)

print("🔧 DECORATED PIPELINE")
print("=" * 50)
raw = extract_orders()
clean = transform_orders(raw)
count = load_orders(clean)
print(f"\n  Pipeline complete: {count} records processed")


## 3.6 File & Path Handling

**WHY:** Data Engineers work with file paths constantly — data lake paths,
config files, log files. `pathlib` is the modern way to handle paths.

In [0]:
# pathlib basics
from pathlib import Path

# Build paths (works cross-platform)
base = Path("/mnt/data")
raw_path = base / "raw" / "orders" / "2024-06-01"
processed_path = base / "processed" / "orders"

print(f"Raw path: {raw_path}")
print(f"Parent: {raw_path.parent}")
print(f"Name: {raw_path.name}")

# File path manipulation
file_path = Path("/mnt/data/raw/orders_20240601.parquet")
print(f"\nFile: {file_path.name}")
print(f"Stem: {file_path.stem}")
print(f"Suffix: {file_path.suffix}")
print(f"Parts: {file_path.parts}")

In [0]:
# JSON config handling
import json

# Write a pipeline config
config = {
    "pipeline": "orders_etl",
    "version": "2.1",
    "source": {"type": "delta", "table": "techmart_dw.orders"},
    "target": {"type": "delta", "table": "techmart_dw.gold_orders"},
    "schedule": "0 6 * * *",
    "quality_rules": [
        {"column": "order_id", "check": "not_null"},
        {"column": "total_amount", "check": "positive"}
    ]
}

# Serialize to JSON string (simulating file write)
config_json = json.dumps(config, indent=2)
print("Pipeline config:")
print(config_json)

# Parse it back
parsed = json.loads(config_json)
print(f"\nPipeline: {parsed['pipeline']} v{parsed['version']}")
print(f"Rules: {len(parsed['quality_rules'])} quality checks")

---
# 📗 Section 4: Python + Spark Integration

## 4.1 Reading Tables & Extracting Data

**WHY:** The bridge between Spark's distributed processing and Python's flexibility.
Know when to use each.

In [0]:
# spark.table() vs spark.sql()
# Both return DataFrames — use whichever is clearer

# Method 1: spark.table()
customers_df = spark.table("customers")
print(f"customers_df type: {type(customers_df)}")
print(f"Columns: {customers_df.columns[:5]}")

# Method 2: spark.sql()
result_df = spark.sql("SELECT customer_segment, COUNT(*) AS cnt FROM customers GROUP BY customer_segment")
result_df.show()

In [0]:
# .collect() — bring data to Python (SMALL DATA ONLY!)
# ⚠️ WARNING: Never collect() large datasets — will crash the driver

# Safe: collecting aggregated results
segments = spark.sql("""
    SELECT customer_segment, COUNT(*) AS cnt, ROUND(AVG(lifetime_value), 2) AS avg_ltv
    FROM customers GROUP BY customer_segment ORDER BY cnt DESC
""").collect()

# Now it's a Python list of Row objects
print(f"Type: {type(segments)}")
print(f"First row: {segments[0]}")
print(f"Access by name: segment={segments[0].customer_segment}, count={segments[0].cnt}")

# Convert to Python dict
segment_dict = {row.customer_segment: row.cnt for row in segments}
print(f"\nAs dict: {segment_dict}")

In [0]:
# Extract a Python list from Spark
emails = spark.sql("""
    SELECT email FROM customers WHERE email IS NOT NULL LIMIT 10
""").collect()

email_list = [row.email for row in emails]
print(f"Email list ({len(email_list)} items):")
for e in email_list:
    print(f"  {e}")

## 4.2 Python Logic with Spark Data

**WHY:** Python orchestrates, Spark executes. Use Python for logic, loops, and configs.
Use Spark for heavy data processing.

In [0]:
# Dynamic query generation with Python
def generate_quality_report(table_name: str) -> dict:
    """Generate a comprehensive quality report for any table."""
    df = spark.table(table_name)
    report = {"table": table_name, "row_count": df.count(), "columns": {}}
    
    for field in df.schema.fields:
        col_name = field.name
        col_type = str(field.dataType)
        
        # Get null count
        null_count = df.filter(f"{col_name} IS NULL").count()
        null_pct = round(null_count / report["row_count"] * 100, 2) if report["row_count"] > 0 else 0
        
        report["columns"][col_name] = {
            "type": col_type,
            "null_count": null_count,
            "null_pct": null_pct
        }
    
    return report

# Generate report for a table
report = generate_quality_report("customers")
print(f"Quality Report: {report['table']} ({report['row_count']:,} rows)")
print(f"\nColumns with NULLs:")
for col, stats in report["columns"].items():
    if stats["null_pct"] > 0:
        print(f"  {col}: {stats['null_count']:,} nulls ({stats['null_pct']}%)")

In [0]:
# Parameterized pipeline function
def run_incremental_load(source_table: str, target_table: str, 
                         watermark_col: str, batch_date: str) -> dict:
    """Run an incremental load from source to target."""
    # Count new records
    count_query = f"""
        SELECT COUNT(*) AS cnt FROM {source_table}
        WHERE {watermark_col} >= '{batch_date}'
    """
    new_count = spark.sql(count_query).collect()[0].cnt
    
    # Build and execute the load (just counting here for demo)
    result = {
        "source": source_table,
        "target": target_table,
        "batch_date": batch_date,
        "new_records": new_count,
        "status": "success" if new_count > 0 else "no_data"
    }
    
    print(f"Pipeline: {source_table} → {target_table}")
    print(f"  Batch date: {batch_date}")
    print(f"  New records: {new_count:,}")
    print(f"  Status: {result['status']}")
    return result

# Run for multiple tables
tables_config = [
    ("orders", "silver_orders", "created_at", "2024-01-01"),
    ("payments", "silver_payments", "created_at", "2024-01-01"),
]

print("=" * 50)
for source, target, wm, dt in tables_config:
    run_incremental_load(source, target, wm, dt)
    print()

In [0]:
# ============================================
# 🎯 YOUR TURN: Table Comparison Function
# ============================================
# Write a function compare_tables(table_a, table_b, key_column) that:
# 1. Gets row counts for both tables
# 2. Finds records in A but not in B (using SQL EXCEPT or LEFT JOIN)
# 3. Finds records in B but not in A
# 4. Returns a dict with counts and status
#
# Test: compare "customers" with "customer_360" on "customer_id"
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
def compare_tables(table_a: str, table_b: str, key_column: str) -> dict:
    count_a = spark.sql(f"SELECT COUNT(*) FROM {table_a}").collect()[0][0]
    count_b = spark.sql(f"SELECT COUNT(*) FROM {table_b}").collect()[0][0]
    
    only_in_a = spark.sql(f"""
        SELECT COUNT(*) FROM {table_a} a
        LEFT JOIN {table_b} b ON a.{key_column} = b.{key_column}
        WHERE b.{key_column} IS NULL
    """).collect()[0][0]
    
    only_in_b = spark.sql(f"""
        SELECT COUNT(*) FROM {table_b} b
        LEFT JOIN {table_a} a ON b.{key_column} = a.{key_column}
        WHERE a.{key_column} IS NULL
    """).collect()[0][0]
    
    result = {
        "table_a": table_a, "table_b": table_b,
        "count_a": count_a, "count_b": count_b,
        "only_in_a": only_in_a, "only_in_b": only_in_b,
        "match": only_in_a == 0 and only_in_b == 0
    }
    
    print(f"Comparison: {table_a} vs {table_b} (key: {key_column})")
    print(f"  {table_a}: {count_a:,} rows")
    print(f"  {table_b}: {count_b:,} rows")
    print(f"  Only in {table_a}: {only_in_a:,}")
    print(f"  Only in {table_b}: {only_in_b:,}")
    print(f"  Match: {'✅' if result['match'] else '❌'}")
    return result

compare_tables("customers", "customer_360", "customer_id")

### Python + Spark: Common Patterns

When working with PySpark, you'll constantly switch between Python and Spark:

In [0]:
# Pattern: Python config → Spark execution
# This is how production pipelines are structured

# Configuration (Python)
pipeline_config = {
    "source_table": "techmart_dw.orders",
    "target_table": "techmart_dw.silver_orders",
    "watermark_column": "order_date",
    "quality_threshold": 0.99,
    "batch_date": "2024-03-15",
}

# Build dynamic SQL from config (Python string formatting → Spark execution)
def build_incremental_query(config):
    """Generate SQL dynamically based on configuration."""
    return f"""
    SELECT *
    FROM {config['source_table']}
    WHERE {config['watermark_column']} >= '{config['batch_date']}'
    """

# In production, you'd execute this:
# df = spark.sql(build_incremental_query(pipeline_config))

query = build_incremental_query(pipeline_config)
print("📝 DYNAMIC SQL GENERATION")
print("=" * 50)
print(f"Config: {pipeline_config}")
print(f"\nGenerated SQL:")
print(query)

print("\n💡 This pattern separates CONFIGURATION from LOGIC.")
print("   Change the config → different behavior, same code.")


In [0]:
# Pattern: Python for complex logic, Spark for data processing
# UDFs (User Defined Functions) bridge Python ↔ Spark

# Example: Complex business logic that's hard to express in SQL
def categorize_customer(lifetime_value, order_count, days_since_last_order):
    """
    Complex customer segmentation logic.
    Too complex for SQL CASE WHEN, so we use a Python UDF.
    """
    if lifetime_value is None or order_count is None:
        return "Unknown"
    
    # VIP: high value + frequent + recent
    if lifetime_value > 5000 and order_count > 20 and days_since_last_order < 30:
        return "VIP"
    
    # At Risk: was valuable but hasn't ordered recently
    if lifetime_value > 2000 and days_since_last_order > 90:
        return "At Risk"
    
    # New: few orders, recent
    if order_count < 3 and days_since_last_order < 60:
        return "New"
    
    # Loyal: consistent ordering
    if order_count > 10 and days_since_last_order < 60:
        return "Loyal"
    
    # Churned: hasn't ordered in a long time
    if days_since_last_order > 180:
        return "Churned"
    
    return "Regular"


# Test the logic in pure Python first (easier to debug!)
test_cases = [
    (10000, 50, 5, "VIP"),
    (8000, 25, 120, "At Risk"),
    (100, 1, 10, "New"),
    (3000, 15, 30, "Loyal"),
    (500, 5, 200, "Churned"),
    (1000, 8, 45, "Regular"),
]

print("🏷️ CUSTOMER SEGMENTATION UDF")
print("=" * 50)
print(f"{'LTV':<8} {'Orders':<8} {'Days':<8} {'Expected':<12} {'Actual':<12} {'Match'}")
print("-" * 60)
for ltv, orders, days, expected in test_cases:
    actual = categorize_customer(ltv, orders, days)
    match = "✅" if actual == expected else "❌"
    print(f"{ltv:<8} {orders:<8} {days:<8} {expected:<12} {actual:<12} {match}")

print("\n💡 In PySpark, register this as a UDF:")
print("   from pyspark.sql.functions import udf")
print("   categorize_udf = udf(categorize_customer, StringType())")
print("   df.withColumn('segment', categorize_udf('ltv', 'orders', 'days'))")


---
# 🚀 Section 5: Mini Projects

## Project 1: Configuration-Driven Pipeline

Build a Python class that reads a config dict and executes a parameterized ETL.

In [0]:
class ConfigDrivenPipeline:
    """Execute ETL based on a configuration dictionary."""
    
    def __init__(self, config: dict):
        self.config = config
        self.name = config["name"]
        self.source = config["source"]
        self.target = config["target"]
        self.transforms = config.get("transforms", [])
        self.metrics = {"rows_read": 0, "rows_written": 0, "errors": 0}
    
    def extract(self) -> int:
        """Read from source and return row count."""
        query = f"SELECT COUNT(*) FROM {self.source['table']}"
        if "filter" in self.source:
            query = f"SELECT COUNT(*) FROM {self.source['table']} WHERE {self.source['filter']}"
        count = spark.sql(query).collect()[0][0]
        self.metrics["rows_read"] = count
        return count
    
    def transform(self) -> list:
        """Apply transformations (returns SQL expressions)."""
        applied = []
        for t in self.transforms:
            applied.append(f"Applied: {t['type']} on {t.get('column', 'all')}")
        return applied
    
    def load(self) -> str:
        """Simulate loading to target."""
        self.metrics["rows_written"] = self.metrics["rows_read"]
        return f"Loaded {self.metrics['rows_written']:,} rows to {self.target['table']}"
    
    def run(self) -> dict:
        """Execute the full pipeline."""
        print(f"🚀 Running pipeline: {self.name}")
        print(f"   Source: {self.source['table']}")
        
        self.extract()
        print(f"   Extracted: {self.metrics['rows_read']:,} rows")
        
        transforms = self.transform()
        for t in transforms:
            print(f"   {t}")
        
        result = self.load()
        print(f"   {result}")
        print(f"   ✅ Pipeline complete!")
        return self.metrics

# Define and run a pipeline
config = {
    "name": "orders_gold_pipeline",
    "source": {"table": "orders", "filter": "status = 'completed'"},
    "target": {"table": "gold_completed_orders", "mode": "overwrite"},
    "transforms": [
        {"type": "filter_nulls", "column": "customer_id"},
        {"type": "deduplicate", "column": "order_id"},
        {"type": "add_audit_columns"}
    ]
}

pipeline = ConfigDrivenPipeline(config)
metrics = pipeline.run()

## Project 2: Data Profiling Tool

A reusable profiler that works on any table.

In [0]:
class DataProfiler:
    """Profile any table with comprehensive statistics."""
    
    def __init__(self, table_name: str):
        self.table_name = table_name
        self.df = spark.table(table_name)
        self.row_count = self.df.count()
        self.col_count = len(self.df.columns)
    
    def profile_column(self, col_name: str) -> dict:
        """Get detailed stats for a single column."""
        stats = spark.sql(f"""
            SELECT 
                COUNT({col_name}) AS non_null,
                COUNT(*) - COUNT({col_name}) AS nulls,
                COUNT(DISTINCT {col_name}) AS distinct_vals
            FROM {self.table_name}
        """).collect()[0]
        
        return {
            "column": col_name,
            "non_null": stats.non_null,
            "nulls": stats.nulls,
            "null_pct": round(stats.nulls / self.row_count * 100, 2),
            "distinct": stats.distinct_vals,
            "uniqueness": round(stats.distinct_vals / self.row_count * 100, 2)
        }
    
    def full_profile(self, max_cols: int = 10):
        """Profile all columns (limited for performance)."""
        print(f"\n{'='*60}")
        print(f"📊 DATA PROFILE: {self.table_name}")
        print(f"{'='*60}")
        print(f"Rows: {self.row_count:,} | Columns: {self.col_count}")
        print(f"\n{'Column':<20} {'Nulls':<8} {'Null%':<8} {'Distinct':<10} {'Unique%'}")
        print("-" * 60)
        
        for field in self.df.schema.fields[:max_cols]:
            stats = self.profile_column(field.name)
            print(f"{stats['column']:<20} {stats['nulls']:<8} {stats['null_pct']:<8} {stats['distinct']:<10} {stats['uniqueness']}%")
        
        if self.col_count > max_cols:
            print(f"\n... and {self.col_count - max_cols} more columns")
        print(f"{'='*60}")

# Profile the orders table
profiler = DataProfiler("orders")
profiler.full_profile()

## Project 3: Alert System

Monitor data freshness and quality, print alerts.

In [0]:
from datetime import datetime

class DataAlertSystem:
    """Monitor data freshness and quality."""
    
    def __init__(self):
        self.alerts = []
        self.checks_run = 0
    
    def check_freshness(self, table: str, date_col: str, max_hours: int = 24):
        """Alert if data is stale."""
        self.checks_run += 1
        result = spark.sql(f"SELECT MAX({date_col}) AS latest FROM {table}").collect()[0]
        latest = result.latest
        
        if latest is None:
            self.alerts.append({"level": "CRITICAL", "table": table, "msg": f"No data in {date_col}"})
        else:
            print(f"  ℹ️ {table}.{date_col} latest: {latest}")
    
    def check_row_count(self, table: str, min_rows: int):
        """Alert if row count drops below minimum."""
        self.checks_run += 1
        count = spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
        
        if count < min_rows:
            self.alerts.append({
                "level": "WARNING", "table": table,
                "msg": f"Row count {count:,} below minimum {min_rows:,}"
            })
        else:
            print(f"  ✅ {table}: {count:,} rows (min: {min_rows:,})")
    
    def check_null_rate(self, table: str, column: str, max_rate: float = 0.1):
        """Alert if null rate exceeds threshold."""
        self.checks_run += 1
        stats = spark.sql(f"""
            SELECT COUNT(*) AS total, SUM(CASE WHEN {column} IS NULL THEN 1 ELSE 0 END) AS nulls
            FROM {table}
        """).collect()[0]
        
        rate = stats.nulls / stats.total if stats.total > 0 else 0
        if rate > max_rate:
            self.alerts.append({
                "level": "WARNING", "table": table,
                "msg": f"{column} null rate {rate:.1%} exceeds {max_rate:.1%}"
            })
        else:
            print(f"  ✅ {table}.{column}: null rate {rate:.1%} (max: {max_rate:.1%})")
    
    def report(self):
        """Print alert summary."""
        print(f"\n{'='*50}")
        print(f"🔔 ALERT REPORT — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f"{'='*50}")
        print(f"Checks run: {self.checks_run}")
        print(f"Alerts: {len(self.alerts)}")
        
        if self.alerts:
            for alert in self.alerts:
                icon = "🔴" if alert["level"] == "CRITICAL" else "🟡"
                print(f"  {icon} [{alert['level']}] {alert['table']}: {alert['msg']}")
        else:
            print("  🟢 All checks passed!")
        print(f"{'='*50}")

# Run monitoring
monitor = DataAlertSystem()
monitor.check_row_count("orders", min_rows=15000)
monitor.check_row_count("customers", min_rows=5000)
monitor.check_null_rate("customers", "email", max_rate=0.05)
monitor.check_null_rate("customers", "phone", max_rate=0.05)
monitor.check_freshness("orders", "order_date")
monitor.report()

---
# 🏆 Section 6: Interview Questions

## Challenge 1: Flatten Nested Dict
Given a nested dictionary, flatten it with dot-notation keys.

In [0]:
# Challenge 1: Flatten nested dict
def flatten_dict(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    """Flatten a nested dictionary."""
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

# Test
nested = {
    "pipeline": {"name": "orders_etl", "version": "2.0"},
    "source": {"database": "raw", "table": "orders", "format": "delta"},
    "target": {"database": "gold", "table": "fact_orders"}
}

flat = flatten_dict(nested)
print("Flattened:")
for k, v in flat.items():
    print(f"  {k}: {v}")

## Challenge 2: Deduplicate Preserving Order
Remove duplicates from a list while preserving insertion order.

In [0]:
# Challenge 2: Deduplicate preserving order
def dedupe_ordered(items: list) -> list:
    """Remove duplicates while preserving order."""
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

# Test
columns = ["id", "name", "email", "name", "id", "phone", "email", "status"]
print(f"Original: {columns}")
print(f"Deduped:  {dedupe_ordered(columns)}")

## Challenge 3: Group Records by Key
Group a list of records by a specified key (like SQL GROUP BY in Python).

In [0]:
# Challenge 3: Group by key
from collections import defaultdict

def group_by(records: list, key: str) -> dict:
    """Group records by a key field."""
    groups = defaultdict(list)
    for record in records:
        groups[record[key]].append(record)
    return dict(groups)

# Test
transactions = [
    {"account": "A001", "amount": 100, "type": "credit"},
    {"account": "A002", "amount": 200, "type": "debit"},
    {"account": "A001", "amount": 150, "type": "credit"},
    {"account": "A002", "amount": 50, "type": "credit"},
    {"account": "A001", "amount": 75, "type": "debit"},
]

grouped = group_by(transactions, "account")
for account, txns in grouped.items():
    total = sum(t["amount"] for t in txns)
    print(f"  {account}: {len(txns)} transactions, total=${total}")

## Challenge 4: Implement a Simple LRU Cache
Useful for caching expensive query results.

In [0]:
# Challenge 4: Simple LRU Cache
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()
    
    def get(self, key: str):
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        return None
    
    def put(self, key: str, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            evicted = self.cache.popitem(last=False)
            print(f"  Evicted: {evicted[0]}")

# Test
cache = LRUCache(3)
cache.put("query_1", "result_1")
cache.put("query_2", "result_2")
cache.put("query_3", "result_3")
print(f"Get query_1: {cache.get('query_1')}")
cache.put("query_4", "result_4")  # Should evict query_2
print(f"Get query_2: {cache.get('query_2')}")  # None (evicted)
print(f"Cache keys: {list(cache.cache.keys())}")

## Challenge 5: Parse and Validate Schema
Validate that incoming data matches an expected schema.

In [0]:
# Challenge 5: Schema validation
def validate_schema(records: list, schema: dict) -> tuple:
    """Validate records against a schema definition.
    
    Schema format: {"column_name": {"type": type, "required": bool}}
    """
    valid, invalid = [], []
    
    for i, record in enumerate(records):
        errors = []
        
        # Check required fields
        for col, rules in schema.items():
            if rules.get("required") and col not in record:
                errors.append(f"Missing required field: {col}")
            elif col in record and record[col] is not None:
                expected_type = rules["type"]
                if not isinstance(record[col], expected_type):
                    errors.append(f"{col}: expected {expected_type.__name__}, got {type(record[col]).__name__}")
        
        if errors:
            invalid.append({"index": i, "record": record, "errors": errors})
        else:
            valid.append(record)
    
    return valid, invalid

# Test
schema = {
    "id": {"type": int, "required": True},
    "name": {"type": str, "required": True},
    "amount": {"type": float, "required": True},
    "status": {"type": str, "required": False}
}

test_data = [
    {"id": 1, "name": "Order A", "amount": 150.0, "status": "active"},
    {"id": "2", "name": "Order B", "amount": 200.0},  # id wrong type
    {"id": 3, "amount": 100.0},  # missing name
    {"id": 4, "name": "Order D", "amount": 300.0, "status": "done"},
]

valid, invalid = validate_schema(test_data, schema)
print(f"Valid: {len(valid)}, Invalid: {len(invalid)}")
for inv in invalid:
    print(f"  ❌ Record {inv['index']}: {inv['errors']}")

---
# ✅ Notebook Complete!

**What was covered:**
- Python fundamentals: variables, strings, lists, dicts, tuples, sets
- Control flow: conditionals, loops, comprehensions
- Functions: type hints, *args/**kwargs, docstrings
- Intermediate: lambdas, error handling, OOP, generators, decorators
- File handling: pathlib, JSON
- Spark integration: spark.table(), spark.sql(), .collect()
- 3 mini projects: Config Pipeline, Data Profiler, Alert System
- 5 interview challenges

**Key takeaway:** Python orchestrates, Spark executes. Use Python for logic,
configuration, and control flow. Use Spark for heavy data processing.

**Next:** Notebook 03 — PySpark Deep Dive

---
*The `techmart_dw` database remains available for subsequent notebooks.*